# VYRO guitar detector

Trains the small model that finds a guitar in the camera picture.

1. Runtime > Change runtime type > **T4 GPU**.
2. Runtime > **Run all**. If Colab asks to restart the session, restart it and run the rest.
3. The GPU cell must print a GPU. Training takes about one to two hours.
4. At the end the browser downloads `model.tflite` and `metrics.json`.

The pictures are public photos from Open Images and stay in this Colab machine
(see `docs/OBJECT_DATA.md`).

In [ ]:
# Part 1 of the guitar detector: the tool that downloads the pictures.
%pip install -q fiftyone

In [ ]:
# Downloads boxes of guitars from Open Images V7 and writes them in the COCO layout that
# MediaPipe Model Maker reads. The pictures stay in this Colab machine; they are never added to
# the VYRO repository. Source and license: docs/OBJECT_DATA.md.
import fiftyone as fo
import fiftyone.zoo as foz

CLASS_NAME = "Guitar"
TRAIN_SAMPLES = 3000
# The test pictures come from the official test split, so the model never sees them while training.
TEST_SAMPLES = 300
TRAIN_DIR = "/content/guitar_data/train"
TEST_DIR = "/content/guitar_data/test"


def fetch(split, max_samples):
    return foz.load_zoo_dataset(
        "open-images-v7",
        split=split,
        label_types=["detections"],
        classes=[CLASS_NAME],
        only_matching=True,
        max_samples=max_samples,
        shuffle=True,
        seed=51,
        dataset_name=f"guitar-{split}",
    )


def export_coco(dataset, folder):
    dataset.export(
        dataset_type=fo.types.COCODetectionDataset,
        data_path=f"{folder}/images",
        labels_path=f"{folder}/labels.json",
        label_field="ground_truth",
        classes=[CLASS_NAME],
    )


train_set = fetch("train", TRAIN_SAMPLES)
test_set = fetch("test", TEST_SAMPLES)
export_coco(train_set, TRAIN_DIR)
export_coco(test_set, TEST_DIR)
print("pictures for training:", len(train_set), "- pictures for the final test:", len(test_set))

In [ ]:
# Part 2: is there a GPU? This must print a table with the name of a GPU (for example a Tesla T4).
# If it says the command was not found, the runtime has no GPU: Runtime > Change runtime type.
!nvidia-smi

In [ ]:
# An environment with Python 3.10 just for training. MediaPipe Model Maker does not install on
# the newer Python that Colab ships, so it lives apart from the rest. It needs setuptools (the
# new environment has none) and the CUDA libraries that TensorFlow uses to reach the GPU.
%pip install -q uv
!uv venv --clear --python 3.10 /content/mm
!uv pip install --python /content/mm/bin/python mediapipe-model-maker "mediapipe==0.10.14" "setuptools<81"
!uv pip install --python /content/mm/bin/python --reinstall-package tensorflow "tensorflow[and-cuda]==2.15.1"

In [ ]:
# The training must see the GPU; on the CPU alone it would take many hours.
!MPLBACKEND=Agg /content/mm/bin/python -c "import tensorflow as tf; print('GPUs:', tf.config.list_physical_devices('GPU'))"

In [ ]:
%%writefile /content/train_model.py
"""Trains the guitar detector with MediaPipe Model Maker and measures it on pictures it never saw.

It runs inside the Python 3.10 environment of the notebook, after download_data.py.
"""

import json
from datetime import date

import tensorflow as tf
from mediapipe_model_maker import object_detector

if not tf.config.list_physical_devices("GPU"):
    raise SystemExit("No GPU found: on the CPU this would take many hours. Stopping.")

CLASS_NAME = "Guitar"
TRAIN_DIR = "/content/guitar_data/train"
TEST_DIR = "/content/guitar_data/test"
EXPORT_DIR = "/content/guitar_model"
# The model is kept only if it finds at least half of the guitars with a good box (AP at IoU 0.5).
MIN_AP50 = 0.5

train_and_validation = object_detector.Dataset.from_coco_folder(
    TRAIN_DIR, cache_dir="/content/cache/train"
)
train_data, validation_data = train_and_validation.split(0.9)
test_data = object_detector.Dataset.from_coco_folder(TEST_DIR, cache_dir="/content/cache/test")

options = object_detector.ObjectDetectorOptions(
    supported_model=object_detector.SupportedModels.MOBILENET_V2,
    hparams=object_detector.HParams(
        batch_size=8, learning_rate=0.3, epochs=30, export_dir=EXPORT_DIR
    ),
)
model = object_detector.ObjectDetector.create(
    train_data=train_data, validation_data=validation_data, options=options
)

# The model is exported first, so a mistake while writing the report cannot lose the training.
model.export_model()


def plain(value):
    """Turns a number, or a list of numbers, into something json can write."""
    if isinstance(value, (list, tuple)):
        return [plain(item) for item in value]
    return float(value)


loss, coco_metrics = model.evaluate(test_data, batch_size=4)
ap50 = float(coco_metrics.get("AP50", 0.0))
report = {
    "date": date.today().isoformat(),
    "class": CLASS_NAME,
    "source": "Open Images V7",
    "base_model": "MOBILENET_V2",
    "train_pictures": train_and_validation.size,
    "test_pictures": test_data.size,
    "loss": plain(loss),
    "coco_metrics": {name: plain(value) for name, value in coco_metrics.items()},
    "minimum_ap50": MIN_AP50,
    "passes": ap50 >= MIN_AP50,
}
print(json.dumps(report, indent=2))

with open(f"{EXPORT_DIR}/metrics.json", "w") as handle:
    json.dump(report, handle, indent=2)


In [ ]:
!MPLBACKEND=Agg /content/mm/bin/python /content/train_model.py

In [ ]:
# Hands the model and its numbers to the browser's downloads.
from google.colab import files

files.download("/content/guitar_model/model.tflite")
files.download("/content/guitar_model/metrics.json")